# ASSET Final Statistical Closure — R4C

This notebook performs the **last analysis-only statistical closure** for the Paper .

It evaluates two frozen paired contrasts:

1. **E3 attention vs frozen B2 no-attention**
2. **E3 vs frozen S2 HuBERT**

No model is trained, tuned, selected, refrozen, or modified.

### Cluster units
- E-SERAVD internal: frozen `source_group_code`
- RAVDESS: canonical `speaker_id`
- CREMA-D: canonical `speaker_id`
- EMO-DB: canonical `speaker_id`
- MALEO: **descriptive only**, because no defensible repeated speaker/group identifier is available in the frozen canonical metadata.

### Inference
- Effect = mean of the three seed-specific paired metric differences.
- 10,000 cluster bootstrap resamples.
- Exact cluster permutation when the number of clusters ≤ 12; otherwise 10,000 Monte-Carlo cluster swaps.
- Holm correction across the **four cluster-identifiable evaluation sets**, separately for each contrast and metric.
- Metrics are computed over the fixed six-class label set.

This implementation is optimized by resampling **cluster-level confusion matrices**, which is mathematically equivalent to resampling all utterances belonging to selected clusters but much faster.

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Imports and frozen settings
from pathlib import Path
import os, json, math
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path('/content/drive/MyDrive/project_thesis_final')
REPS = 10_000
RNG_SEED = 4696
SEEDS = (42, 52, 62)
LABELS = np.arange(6, dtype=int)

assert PROJECT_DIR.exists(), PROJECT_DIR

FINAL_ROOT = PROJECT_DIR / 'SER_EXPERIMENTS_V3/campaigns/final_E1_E8'
NOATT_ROOT = PROJECT_DIR / 'SER_EXPERIMENTS_V3/reporting/asset_no_attention_external_v1'
SSL_ROOT = PROJECT_DIR / 'SER_EXPERIMENTS_SSL_V1'
MANIFEST = PROJECT_DIR / 'DATA_CANONICAL_V2/manifests/analysis_manifest_final.csv'
SPLIT = PROJECT_DIR / 'SER_EXPERIMENTS_V2/splits/eseravd_group_disjoint_split_v1.csv'

for p in [FINAL_ROOT, NOATT_ROOT, SSL_ROOT, MANIFEST, SPLIT]:
    assert p.exists(), f'Missing required frozen artifact: {p}'

OUT_DIR = PROJECT_DIR / 'SER_EXPERIMENTS_V3/reporting/asset_final_cluster_closure_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output:', OUT_DIR)

In [ ]:
# 3) Resolve metadata and cluster units
manifest = pd.read_csv(MANIFEST, low_memory=False)
split = pd.read_csv(SPLIT, low_memory=False)

meta = {}

s = split[split['split'].astype(str).eq('test')].copy().reset_index(drop=True)
s['class_index'] = s['label_id'].astype(int) - 1
meta['eseravd_internal_test'] = (s, 'source_group_code')

for ds in ['maleo', 'ravdess', 'cremad', 'emodb']:
    d = manifest[manifest['dataset'].astype(str).str.lower().eq(ds)].copy().reset_index(drop=True)
    d['class_index'] = d['label_id'].astype(int) - 1
    cluster_col = None if ds == 'maleo' else 'speaker_id'
    if cluster_col is not None:
        assert cluster_col in d.columns
        assert not d[cluster_col].isna().any(), f'{ds}: missing {cluster_col}'
    meta[ds] = (d, cluster_col)

audit_rows = []
for ev, (d, col) in meta.items():
    audit_rows.append({
        'evaluation_set': ev,
        'n_utterances': len(d),
        'cluster_field': col if col else 'NONE — descriptive only',
        'n_clusters': d[col].astype(str).nunique() if col else np.nan,
        'all_six_classes': set(d['class_index'].unique()) == set(range(6)),
    })

display(pd.DataFrame(audit_rows))

In [ ]:
# 4) Locate frozen prediction files

def e3_path(seed, ev):
    return FINAL_ROOT / f'E3_seed{seed}' / 'results' / f'{ev}_predictions.csv'

def noatt_path(seed, ev):
    return NOATT_ROOT / f'B2_no_attention_seed{seed}_{ev}_predictions.csv'

def hubert_path(seed, ev):
    target = f'{ev}_ssl_predictions.csv'
    # Search only under the frozen SSL root and require S2_seed{seed} in the path.
    matches = [
        p for p in SSL_ROOT.rglob(target)
        if f'S2_seed{seed}' in p.parts
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f'HuBERT path resolution for seed={seed}, ev={ev}: '
            f'expected exactly 1 match, found {len(matches)}: {matches}'
        )
    return matches[0]

rows = []
for seed in SEEDS:
    for ev in meta:
        for system, fn in [
            ('E3', e3_path),
            ('NoAttention', noatt_path),
            ('HuBERT', hubert_path),
        ]:
            p = fn(seed, ev)
            rows.append({'seed': seed, 'evaluation_set': ev, 'system': system,
                         'exists': p.exists(), 'path': str(p)})

path_audit = pd.DataFrame(rows)
assert path_audit['exists'].all(), path_audit[~path_audit.exists].to_string(index=False)
display(path_audit.groupby('system')['exists'].agg(['count','all']))
print('✓ All frozen prediction files located.')

In [ ]:
# 5) Load and validate frozen predictions against canonical row order

def load_one(path, metadata):
    df = pd.read_csv(path)
    required = {'true_class_index', 'pred_class_index'}
    assert required.issubset(df.columns), f'{path}: missing {required - set(df.columns)}'
    assert len(df) == len(metadata), f'{path}: {len(df)} != metadata {len(metadata)}'
    y = df['true_class_index'].to_numpy(int)
    p = df['pred_class_index'].to_numpy(int)
    expected = metadata['class_index'].to_numpy(int)
    assert np.array_equal(y, expected), f'{path}: y_true/order mismatch'
    assert set(np.unique(y)).issubset(set(range(6)))
    assert set(np.unique(p)).issubset(set(range(6)))
    return y, p

preds = {}
for ev, (mdf, _) in meta.items():
    preds[ev] = {}
    for seed in SEEDS:
        preds[ev][seed] = {}
        preds[ev][seed]['E3'] = load_one(e3_path(seed, ev), mdf)
        preds[ev][seed]['NoAttention'] = load_one(noatt_path(seed, ev), mdf)
        preds[ev][seed]['HuBERT'] = load_one(hubert_path(seed, ev), mdf)

print('✓ All prediction rows/labels match canonical metadata.')

In [ ]:
# 6) Metric and cluster-confusion helpers

def confusion6(y, p):
    cm = np.zeros((6,6), dtype=np.int64)
    np.add.at(cm, (y, p), 1)
    return cm

def score_from_cm(cm, metric):
    cm = np.asarray(cm, dtype=float)

    # IMPORTANT:
    # cm can be either one 6x6 confusion matrix OR a batch shaped (..., 6, 6).
    # np.diag() only supports 1-D/2-D arrays, so use np.diagonal over
    # the last two axes to support vectorized bootstrap/permutation batches.
    tp = np.diagonal(cm, axis1=-2, axis2=-1)

    row = cm.sum(axis=-1)   # true-class support, shape (..., 6)
    col = cm.sum(axis=-2)   # predicted-class support, shape (..., 6)

    if metric == 'uar':
        recall = np.divide(
            tp, row,
            out=np.zeros_like(tp, dtype=float),
            where=row > 0
        )
        return recall.mean(axis=-1)

    if metric == 'macro_f1':
        denom = 2*tp + (col - tp) + (row - tp)
        f1 = np.divide(
            2*tp, denom,
            out=np.zeros_like(tp, dtype=float),
            where=denom > 0
        )
        return f1.mean(axis=-1)

    raise KeyError(metric)

def build_cluster_cms(ev, system):
    mdf, cluster_col = meta[ev]
    assert cluster_col is not None
    clusters = mdf[cluster_col].astype(str).to_numpy()
    uniq = np.array(pd.unique(clusters), dtype=object)

    # [seed, cluster, true, pred]
    arr = np.zeros((len(SEEDS), len(uniq), 6, 6), dtype=np.int64)

    for si, seed in enumerate(SEEDS):
        y, p = preds[ev][seed][system]
        for gi, g in enumerate(uniq):
            idx = np.flatnonzero(clusters == g)
            arr[si, gi] = confusion6(y[idx], p[idx])

    return uniq, arr

def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    out = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        candidate = (m-rank) * p[idx]
        running = max(running, candidate)
        out[idx] = min(1.0, running)
    return out

In [ ]:
# 6B) Vectorization self-test — must pass before statistical analysis
_test_single = np.eye(6, dtype=int)
_test_batch = np.stack([_test_single, _test_single], axis=0)

assert np.isclose(score_from_cm(_test_single, 'uar'), 1.0)
assert np.allclose(score_from_cm(_test_batch, 'uar'), [1.0, 1.0])
assert np.isclose(score_from_cm(_test_single, 'macro_f1'), 1.0)
assert np.allclose(score_from_cm(_test_batch, 'macro_f1'), [1.0, 1.0])

print('✓ Vectorized confusion-matrix metric self-test passed.')

In [ ]:
# 7) Optimized cluster-aware paired inference

def run_cluster_contrast(ev, system_b, metric, reps=REPS, rng_seed=RNG_SEED):
    # Delta = E3 - comparator
    uniq, A = build_cluster_cms(ev, 'E3')
    uniq_b, B = build_cluster_cms(ev, system_b)
    assert np.array_equal(uniq, uniq_b)
    k = len(uniq)

    # Observed mean paired effect across seeds.
    cmA_total = A.sum(axis=1)   # [seed,6,6]
    cmB_total = B.sum(axis=1)
    observed = float(np.mean(score_from_cm(cmA_total, metric) - score_from_cm(cmB_total, metric)))

    # Reproducible RNG specific to eval/system/metric.
    offset = sum(map(ord, ev + system_b + metric))
    rng = np.random.default_rng(rng_seed + offset)

    # Cluster bootstrap: Multinomial counts are exactly equivalent to
    # drawing k clusters with replacement k times.
    weights = rng.multinomial(k, np.repeat(1.0/k, k), size=reps)  # [reps,k]
    boot_seed = []
    for si in range(len(SEEDS)):
        cmA = np.einsum('rk,kij->rij', weights, A[si], optimize=True)
        cmB = np.einsum('rk,kij->rij', weights, B[si], optimize=True)
        boot_seed.append(score_from_cm(cmA, metric) - score_from_cm(cmB, metric))
    boot = np.mean(np.stack(boot_seed, axis=0), axis=0)
    ci_low, ci_high = np.quantile(boot, [0.025,0.975])

    # Cluster permutation: swap entire clusters for E3/comparator using the
    # same swap mask for all three model seeds.
    if k <= 12:
        n_perm = 1 << k
        masks = ((np.arange(n_perm)[:,None] >> np.arange(k)) & 1).astype(float)
        method = f'exact_{n_perm}'
    else:
        masks = (rng.random((reps,k)) < 0.5).astype(float)
        n_perm = reps
        method = f'monte_carlo_{reps}'

    null_seed = []
    for si in range(len(SEEDS)):
        # Base totals + swaps*(other-own)
        baseA = A[si].sum(axis=0)
        baseB = B[si].sum(axis=0)
        deltaAB = B[si] - A[si]
        swap_delta = np.einsum('rk,kij->rij', masks, deltaAB, optimize=True)
        cmA = baseA[None,:,:] + swap_delta
        cmB = baseB[None,:,:] - swap_delta
        null_seed.append(score_from_cm(cmA, metric) - score_from_cm(cmB, metric))

    null = np.mean(np.stack(null_seed, axis=0), axis=0)

    if k <= 12:
        p = float(np.mean(np.abs(null) >= abs(observed) - 1e-15))
    else:
        p = float((1 + np.sum(np.abs(null) >= abs(observed) - 1e-15)) / (len(null) + 1))

    return {
        'evaluation_set': ev,
        'contrast': f'E3-{system_b}',
        'metric': metric,
        'n_clusters': k,
        'delta': observed,
        'ci_low': float(ci_low),
        'ci_high': float(ci_high),
        'cluster_permutation_p_raw': p,
        'permutation_method': method,
        'bootstrap_reps': reps,
    }

cluster_evals = ['eseravd_internal_test','ravdess','cremad','emodb']
comparators = ['NoAttention','HuBERT']
metrics = ['uar','macro_f1']

results = []
for system_b in comparators:
    for ev in cluster_evals:
        for metric in metrics:
            print(f'Running {ev:22s} E3 vs {system_b:11s} {metric}...')
            results.append(run_cluster_contrast(ev, system_b, metric))

res = pd.DataFrame(results)

# Holm within each contrast × metric family across the four cluster-identifiable sets.
res['holm_p_4_cluster_sets'] = np.nan
for (contrast, metric), idx in res.groupby(['contrast','metric']).groups.items():
    res.loc[idx, 'holm_p_4_cluster_sets'] = holm_adjust(
        res.loc[idx, 'cluster_permutation_p_raw'].to_numpy(float)
    )

res['holm_significant_0_05'] = res['holm_p_4_cluster_sets'] < 0.05
res['cluster_ci_excludes_zero'] = (res['ci_low'] > 0) | (res['ci_high'] < 0)

display(res)

In [ ]:
# 8) Add MALEO as descriptive-only evidence

def metric_from_arrays(y,p,metric):
    return float(score_from_cm(confusion6(y,p), metric))

maleo_rows = []
ev = 'maleo'
for system_b in comparators:
    for metric in metrics:
        seed_deltas = []
        e3_scores = []
        b_scores = []
        for seed in SEEDS:
            y,a = preds[ev][seed]['E3']
            _,b = preds[ev][seed][system_b]
            sa = metric_from_arrays(y,a,metric)
            sb = metric_from_arrays(y,b,metric)
            e3_scores.append(sa)
            b_scores.append(sb)
            seed_deltas.append(sa-sb)

        maleo_rows.append({
            'evaluation_set':'maleo',
            'contrast':f'E3-{system_b}',
            'metric':metric,
            'n_clusters':np.nan,
            'E3_mean':np.mean(e3_scores),
            'comparator_mean':np.mean(b_scores),
            'delta':np.mean(seed_deltas),
            'delta_sd_across_seeds':np.std(seed_deltas,ddof=1),
            'inference':'DESCRIPTIVE ONLY — no defensible cluster identifier',
        })

maleo = pd.DataFrame(maleo_rows)
display(maleo)

In [ ]:
# 9) Reviewer-facing compact tables

attention = res[res['contrast'].eq('E3-NoAttention')].copy()
hubert = res[res['contrast'].eq('E3-HuBERT')].copy()

display(attention[[
    'evaluation_set','metric','n_clusters','delta','ci_low','ci_high',
    'holm_p_4_cluster_sets','holm_significant_0_05','cluster_ci_excludes_zero'
]].sort_values(['metric','evaluation_set']))

display(hubert[[
    'evaluation_set','metric','n_clusters','delta','ci_low','ci_high',
    'holm_p_4_cluster_sets','holm_significant_0_05','cluster_ci_excludes_zero'
]].sort_values(['metric','evaluation_set']))

print('\nInterpretation guardrails:')
print('1. Positive Δ favors E3.')
print('2. Statistical significance is defined by Holm-adjusted cluster-permutation p < .05.')
print('3. Bootstrap CI is effect-size uncertainty; it does not override the permutation decision.')
print('4. E-SERAVD internal has only two held-out source groups: coarse sensitivity only.')
print('5. MALEO remains descriptive only.')

In [ ]:
# 10) Save frozen closure pack

res.to_csv(OUT_DIR / 'ASSET_final_cluster_closure_inference.csv', index=False)
maleo.to_csv(OUT_DIR / 'ASSET_final_cluster_closure_MALEO_descriptive.csv', index=False)

attention.to_csv(OUT_DIR / 'ASSET_attention_cluster_aware.csv', index=False)
hubert.to_csv(OUT_DIR / 'ASSET_HuBERT_cluster_aware.csv', index=False)

protocol = {
    'analysis': 'ASSET final cluster-aware closure',
    'no_training': True,
    'seeds': list(SEEDS),
    'bootstrap_reps': REPS,
    'cluster_evaluation_sets': cluster_evals,
    'maleo_cluster_inference': False,
    'delta_definition': 'E3 minus comparator',
    'holm_family': 'four cluster-identifiable evaluation sets, separately per contrast and metric',
    'fixed_six_class_metrics': True,
    'optimization': 'cluster-level confusion-matrix resampling; equivalent to resampling all utterances in selected clusters',
}
(OUT_DIR / 'PROTOCOL.json').write_text(json.dumps(protocol, indent=2))

readme = '''# ASSET Final Cluster-Aware Closure v1

Analysis only. No model was trained, tuned, selected, refrozen, or modified.

Contrasts:
- E3 vs frozen B2 no-attention
- E3 vs frozen S2 HuBERT

Cluster units:
- E-SERAVD internal: source_group_code
- RAVDESS / CREMA-D / EMO-DB: speaker_id
- MALEO: descriptive only because no defensible repeated speaker/group identifier is available.

Significance rule:
Holm-adjusted cluster-level paired permutation p < 0.05.
Bootstrap confidence intervals are supporting effect-size uncertainty, not the primary significance decision.

E-SERAVD internal has only two held-out source groups and is therefore a coarse sensitivity analysis.
'''
(OUT_DIR / 'README.md').write_text(readme)

print('✓ Final statistical closure written to:')
print(OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(' -', p.name)